# Prepare the current multi-view cache from saved native regions

The preparation and prediction actions reuse saved LocateAnything boxes and SAM masks without reading anomaly labels. Only the explicit `dev-evaluate` action opens the separately bound development evaluator after complete prediction seals pass. No action trains a scorer or runs the old P1 pipeline. **CPU FP32 DINO** matches the application feature contract. No scientific dataset run has been performed by preparing this notebook.

Supported regions: a sealed08G comparison-region package; one native08E/protected `prediction_manifest.jsonl` plus its verified RGB package; the exact100+600+300 native development components and combined development lock; or a current-native-regions resource. Membership must match an existing1000/600/400 original role or715/2437 official role. Old P1 scores, heatmaps and float16 global embeddings are ignored. Full grid08N features can be reused only when region IDs, crop bytes, CPU runtime, model and config hashes match.

When LA/SAM outputs are missing, the plan records that prerequisite. The separate native-region preparation notebook supplies only missing LA/SAM outputs under the current freeze. Raw archive conversion and anomaly-label/evaluator conversion are outside this notebook. The old end-to-end09A runner is incompatible with the current score contract.

In [ ]:
from pathlib import Path
import os
import json
RUNTIME_PROFILE = None
NOTEBOOK_RELATIVE = 'notebooks/input-preparation/current-native-regions-to-object-cache.ipynb'
try:
    from pathlib import Path, PurePosixPath
    import hashlib
    import importlib.util
    import json
    import os
    import shutil
    import stat
    import sys
    import tempfile
    import zipfile

    # Bounded input lookup supports flat, namespaced and automatically expanded mounts.
    def _children(path):
        if not path.is_dir():
            return []
        children = sorted(p for p in path.iterdir() if p.is_dir())
        if len(children) > 250:
            raise ValueError('Source discovery bound exceeded; set VPI_CODE_ROOT explicitly')
        return children

    def _source_candidates():
        explicit = os.getenv('VPI_CODE_ROOT')
        if explicit:
            return [Path(explicit).expanduser()]
        mount = Path('/kaggle/input')
        roots = [Path.cwd(), *Path.cwd().parents]
        roots += [p for p in _children(mount) if p.name not in {'datasets', 'notebooks', 'models'}]
        for category, depth in (('datasets', 2), ('notebooks', 2), ('models', 4)):
            current = [mount / category]
            for _ in range(depth):
                current = [child for parent in current for child in _children(parent)]
                if len(current) > 1000:
                    raise ValueError('Source mount inventory exceeded its bound')
            for parent in current:
                roots += _children(parent / 'versions') if (parent / 'versions').is_dir() else [parent]
        return [root / suffix for root in roots for suffix in ('', 'code', 'repository-files', 'source')]

    def _archive_source():
        configured = os.getenv('VPI_SOURCE_ARCHIVE')
        if not configured:
            raise FileNotFoundError('Mount the prepared source package or set VPI_CODE_ROOT explicitly')
        archive = Path(configured)
        expected = os.getenv('VPI_SOURCE_ARCHIVE_SHA256')
        if not expected or archive.stat().st_size > 256 * 1024 * 1024 or hashlib.sha256(archive.read_bytes()).hexdigest() != expected:
            raise ValueError('Explicit source archive checksum or size check failed')
        destination = Path(os.getenv('VPI_SOURCE_DEST', '/kaggle/working/current-source')).resolve()
        if Path('/kaggle').is_dir() and not destination.is_relative_to(Path('/kaggle/working')):
            raise ValueError('Extracted source must be under /kaggle/working')
        if destination.exists():
            raise ValueError('Source destination exists; verify it with VPI_CODE_ROOT instead of overwriting')
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = Path(tempfile.mkdtemp(prefix='current-source.partial-', dir=destination.parent))
        try:
            with zipfile.ZipFile(archive) as bundle:
                infos = bundle.infolist()
                names = [i.filename for i in infos]
                if len(names) != len(set(names)) or sum(i.file_size for i in infos) > 256 * 1024 * 1024 or bundle.testzip():
                    raise ValueError('Source archive inventory, size or CRC is invalid')
                for info in infos:
                    name = info.filename
                    path = PurePosixPath(name)
                    if info.is_dir() or path.is_absolute() or '\\' in name or any(p in {'', '.', '..'} for p in name.split('/')) or stat.S_ISLNK(info.external_attr >> 16):
                        raise ValueError('Unsafe source archive member')
                    output = temporary / name
                    output.parent.mkdir(parents=True, exist_ok=True)
                    output.write_bytes(bundle.read(info))
            os.replace(temporary, destination)
        except BaseException:
            shutil.rmtree(temporary)
            raise
        return destination

    _matches = sorted({p.resolve() for p in _source_candidates() if (p / 'prepared_source_manifest.json').is_file()})
    if len(_matches) > 1:
        raise ValueError('Multiple prepared source roots found; set VPI_CODE_ROOT explicitly')
    CODE_ROOT = _matches[0] if _matches else _archive_source()
    _manifest = json.loads((CODE_ROOT / 'prepared_source_manifest.json').read_text())
    # Verify the bootstrap itself before executing any code from the mounted package.
    _helper = CODE_ROOT / 'scripts/current_notebook_runtime.py'
    if _helper.is_symlink() or hashlib.sha256(_helper.read_bytes()).hexdigest() != _manifest.get('files', {}).get('scripts/current_notebook_runtime.py'):
        raise ValueError('Prepared bootstrap source checksum differs')
    _spec = importlib.util.spec_from_file_location('current_notebook_runtime', _helper)
    _runtime = importlib.util.module_from_spec(_spec)
    _spec.loader.exec_module(_runtime)
    BOOTSTRAP = _runtime.bootstrap(CODE_ROOT, NOTEBOOK_RELATIVE, runtime_profile=RUNTIME_PROFILE)
    CODE_PROVENANCE = BOOTSTRAP['provenance']
    ROOT = CODE_ROOT
    # Scientific imports are allowed only after the exact runtime/source preflight.
    from IPython.display import display, Image
    import pandas as pd
    display({'source': CODE_PROVENANCE, 'runtime': BOOTSTRAP['runtime'],
             'preparation_only': 'Verified source/runtime does not establish protected-resource readiness'})
except BaseException as _bootstrap_error:
    _failure_root = Path('/kaggle/working/runtime-receipts') if Path('/kaggle').is_dir() else Path.cwd()/'.local/runtime-receipts'
    _failure_root.mkdir(parents=True, exist_ok=True)
    _failure = {'status':'bootstrap_failed','error_type':type(_bootstrap_error).__name__,'scientific_run_started':False}
    _temporary = _failure_root/'bootstrap-failure.json.partial'
    _temporary.write_text(json.dumps(_failure,sort_keys=True)+'\n')
    os.replace(_temporary,_failure_root/'bootstrap-failure.json')
    raise

## Explicit inputs and prediction boundary

Set `VPI_NEXT_CONFIG` to a private JSON configuration. Required keys: `role`, `region_format`, `region_root` (or `components` plus `full_development_lock`), `expected_region_lock_sha256`, `feature_root`, `neutral_root`, `output_root`, and current freeze/exposure/prerequisite roots. Native prediction manifests also need `rgb_root`. Development components use cohort IDs as keys, each with `region_root` and `rgb_root`.

`neutral_root` contains exactly a sealed `neutral_correspondence.json`: schema `vpi-current-neutral-correspondence/v1`, the exact region-lock hash, provenance `dataset_reference` or `user_confirmed`, and `observations` with only scene, view_id, candidate_id and object_id. Every candidate requires an explicit ID or null; duplicate IDs in one view are rejected. This map supplies identities, never the unusual-object answer.

In [ ]:
CONFIG = _runtime.resolve_config('native-cache', ROOT)
from vpi_next_evaluation.native_cache import plan_native, extract_missing, assemble_native, prepare_shard
ACTION = CONFIG.get('action', 'plan')
PLAN = plan_native(CONFIG, ROOT) if ACTION in {'plan', 'extract_missing', 'assemble', 'prepare_shard'} else {'status':'typed cached development continuation; no region/model work'}
display(PLAN)

## Choose one explicit operation

`action=plan` is the default and performs no neural work. `action=extract_missing` requires `model_source` and the separately pinned DINO environment (torch2.13.0, Transformers5.14.1, NumPy2.5.3, Pillow12.3.0). It encodes only missing automatic-region crops, at most four per batch. Corrupt checkpoints and changed bindings stop. A completed08N cache is reusable; finish an incomplete08N run with its original source or use a fresh feature directory.

`action=assemble` recomputes the fixed forward/P1/P2 components from existing full grids and saves foreground banks as compressed FP32 NPZ, with selected-view lazy loading. It saves per-scene atomic checkpoints, all-candidate geometry and reusable images. No full grids enter JSON. The conservative output budget includes checkpoints and released copies. Set `maximum_output_bytes` only to a verified available volume. `max_new_scenes` supports incremental assembly; a final cache seal requires the complete original cohort.

`remap_audit` verifies the historical08N automatic tensors against the native100+600+300 region lineage. It checks the exact hashed-ID crosswalk, RGB bytes, packed/native and raw/08N mask hashes, crop coordinates and FP32 arrays. `remap_import` copies only the verified automatic NPZ files byte-for-byte into a new full-cohort binding; it never renames tensor indices or changes feature values. The original features remain sealed.

Use `shard_count` and zero-based `shard_index` for deterministic lexical scene partitions. `max_new_units` and `max_new_scenes` bound each continuation. `prepare_shard` performs missing CPU DINO extraction and cache assembly in one job. Set `feature_roots` to immutable compatible mounted reuse shards; `feature_root` remains the writable destination for genuinely missing units. A partial feature directory is exported with `feature_shard_lock.json`; completed scene partitions use `cache_shard_lock.json` and cannot be evaluated independently as a complete cohort.

`merge_cache_shards` accepts `cache_shards` plus the normal role/gate/output configuration. It verifies every shard, rejects overlaps and incomplete original membership, copies only released RGB/foreground tensors and metadata, and emits the final full-cohort seal. It does not copy raw full-grid checkpoints. Size partitions so every job fits its reviewed output limit and the final merge fits the platform input-mount limit; retain all shards before ending each session.

For the complete original1000-scene development role only, `dev-predict` produces both fixed methods over the same nineteen schedules from the typed cache; `dev-evaluate` verifies those predictions before opening the separately bound development evaluator and renders reusable reports. These actions never create calibration/internal-lock protocol receipts or an independent-confirmation claim. They use the same current adapters, without another method search. Configure cache_root; dev-evaluate additionally needs prediction_root, evaluator_root and report_root. Use separate output directories for assembly, prediction, evaluation and reports.

In [ ]:
if ACTION not in {'plan', 'extract_missing', 'assemble', 'prepare_shard', 'remap_audit', 'remap_import', 'merge_cache_shards', 'dev-predict', 'dev-evaluate'}:
    raise ValueError('Unsupported preparation or fixed-development action')
if ACTION == 'extract_missing':
    CONFIG['feature_root'] = str(_runtime.writable_output(CONFIG['feature_root']))
    RESULT = extract_missing(CONFIG, ROOT)
elif ACTION == 'assemble':
    CONFIG['output_root'] = str(_runtime.writable_output(CONFIG['output_root']))
    RESULT = assemble_native(CONFIG, ROOT)
elif ACTION == 'prepare_shard':
    CONFIG['feature_root'] = str(_runtime.writable_output(CONFIG['feature_root']))
    CONFIG['output_root'] = str(_runtime.writable_output(CONFIG['output_root']))
    RESULT = prepare_shard(CONFIG, ROOT)
elif ACTION == 'remap_audit':
    from vpi_next_evaluation.feature_remap import audit_remap
    CONFIG['remap_root'] = str(_runtime.writable_output(CONFIG['remap_root']))
    RESULT = audit_remap(CONFIG)
elif ACTION == 'remap_import':
    from vpi_next_evaluation.feature_remap import import_remapped
    CONFIG['feature_root'] = str(_runtime.writable_output(CONFIG['feature_root']))
    RESULT = import_remapped(CONFIG)
elif ACTION == 'merge_cache_shards':
    from vpi_next_evaluation.cache_shards import merge_cache_shards
    CONFIG['output_root'] = str(_runtime.writable_output(CONFIG['output_root']))
    RESULT = merge_cache_shards(CONFIG, ROOT)
elif ACTION == 'dev-predict':
    CONFIG['output_root'] = str(_runtime.writable_output(CONFIG['output_root']))
    from vpi_next_evaluation.original import predict_current
    RESULT = predict_current('dev-predict', CONFIG, ROOT)
elif ACTION == 'dev-evaluate':
    CONFIG['output_root'] = str(_runtime.writable_output(CONFIG['output_root']))
    CONFIG['report_root'] = str(_runtime.writable_output(CONFIG['report_root']))
    from vpi_next_evaluation.original import evaluate_predictions
    RESULT = evaluate_predictions('dev-evaluate', CONFIG, ROOT)
    from vpi_next_evaluation.reporting import render_current_report
    REPORT = render_current_report({**CONFIG, 'evaluation_root': CONFIG['output_root']})
    display(REPORT)
    for name in REPORT['aggregate_figures']:
        display(Image(filename=str(Path(CONFIG['report_root']) / 'shareable/figures' / name)))
else:
    RESULT = PLAN
display(RESULT)

## Retain outputs and hand off

Keep full DINO grids, atomic checkpoints and the assembled model cache as separate retained resources. The model cache seal includes only its cohort manifest, scene metadata, RGB and compressed foreground tensors. Preparation receipts and recovery state are private. A successful cache conversion is not a completed accuracy evaluation.

Mount only the model cache in09A/09C/10A/10A1. Typed evaluator packages and reviewed asset maps remain separate until complete predictions are sealed. Use the shared verified recovery exporter described in the Kaggle preparation guide; checkpoint files must actually be retained or exported before ending a session. Never rerun neural extraction to repair reports.